In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

summary = pd.read_csv(
    "kyandall_project_summary.csv",
    sep=";"
)

# Unix timestamp -> normal datetime
summary["datetime"] = pd.to_datetime(
    pd.to_numeric(summary["time"]),
    unit="s"
)

# Month representation used throughout project
summary["Month"] = summary["datetime"].dt.to_period("M")

print(summary.shape)
summary.head()

(48907, 7)


,project_wocid,commit_sha1,author,time,commit message,datetime,Month
0,ucl_mimix,00218dd20dbf561c06e03ebc81d99c644358fe01,KMcgrathUCL <45361270+KMcgrathUCL@users.norepl...,1610635036,asthma data ref arm 3,2021-01-14 14:37:16,2021-01
1,ucl_mimix,003e7448982837f6d239e3bc8620fc504e16fca2,KMcgrathUCL <45361270+KMcgrathUCL@users.norepl...,1615551804,edit data example quotes,2021-03-12 12:23:24,2021-03
2,ucl_mimix,009619e8ce4e0e5f24945517015ac4ddd9084a61,KMcgrathUCL <45361270+KMcgrathUCL@users.norepl...,1617101733,Delete pass2Loop.Rd,2021-03-30 10:55:33,2021-03
3,ucl_mimix,00fda073c9f0e598020717e14cb53c186f9de8bb,KMcgrathUCL <45361270+KMcgrathUCL@users.norepl...,1614445850,Delete Runmimix.Rd,2021-02-27 17:10:50,2021-02
4,ucl_mimix,01c93c72f6404a0950d097892849343dd977c057,KMcgrathUCL <45361270+KMcgrathUCL@users.norepl...,1615971260,v0.0.13 edits,2021-03-17 08:54:20,2021-03


In [16]:
monthly_by_project = {}

for project, project_df in summary.groupby("project_wocid"):

    # Count commits in each month
    monthly_counts = project_df.groupby("Month").size()

    # Full range from first commit month to last commit month
    all_months = pd.period_range(
        start=project_df["Month"].min(),
        end=project_df["Month"].max(),
        freq="M"
    )

    # Add missing months with 0 commits
    monthly_counts = monthly_counts.reindex(
        all_months,
        fill_value=0
    )

    timeseries = pd.DataFrame({
        "Month": monthly_counts.index.astype(str),
        "#Commits": monthly_counts.values
    })

    monthly_by_project[project] = timeseries

    # Save monthly CSV for project
    timeseries.to_csv(
        f"kyandall_commits_timeseries_{project}.csv",
        sep=";",
        index=False
    )

    # Plot
    plt.figure(figsize=(12, 5))

    plt.plot(
        range(len(timeseries)),
        timeseries["#Commits"]
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)
    plt.grid(True)

    # Don't print every month or labels become unreadable
    step = max(1, len(timeseries) // 12)

    plt.xticks(
        range(0, len(timeseries), step),
        timeseries["Month"].iloc[::step],
        rotation=45
    )

    plt.tight_layout()

    plt.savefig(
        f"kyandall_timeseries_{project}.png",
        dpi=300
    )

    plt.close()

print("Finished generating time series and plots.")

Finished generating time series and plots.


In [17]:
def analyze_gaps(timeseries):

    zero_runs = []

    start = None

    for i, commits in enumerate(timeseries["#Commits"]):

        if commits == 0:

            if start is None:
                start = i

        else:

            if start is not None:
                zero_runs.append((start, i - 1))
                start = None

    # Gap continues through final month
    if start is not None:
        zero_runs.append((start, len(timeseries) - 1))


    # No zero-commit months
    if len(zero_runs) == 0:
        return {
            "LongestGapStart": None,
            "LongestGapEnd": None,
            "LongestGapLength": 0,
            "NumCommitsAfterLastGap": 0,
            "NumberOfGapsInTimeline": 0
        }


    # Find longest zero run
    longest = max(
        zero_runs,
        key=lambda x: x[1] - x[0] + 1
    )

    start_index, end_index = longest

    gap_length = end_index - start_index + 1


    # Number of gaps that lasted >= 3 months
    num_large_gaps = sum(
        1
        for start, end in zero_runs
        if (end - start + 1) >= 3
    )


    # Number of commits after longest gap
    commits_after = timeseries.loc[
        end_index + 1:,
        "#Commits"
    ].sum()


    return {
        "LongestGapStart":
            timeseries.loc[start_index, "Month"],

        "LongestGapEnd":
            timeseries.loc[end_index, "Month"],

        "LongestGapLength":
            gap_length,

        "NumCommitsAfterLastGap":
            int(commits_after),

        "NumberOfGapsInTimeline":
            num_large_gaps
    }

In [18]:
gap_results = []

for project, timeseries in monthly_by_project.items():

    result = analyze_gaps(timeseries)

    result["Project"] = project

    gap_results.append(result)


gap_df = pd.DataFrame(gap_results)

gap_df = gap_df[
    [
        "Project",
        "LongestGapStart",
        "LongestGapEnd",
        "LongestGapLength",
        "NumCommitsAfterLastGap",
        "NumberOfGapsInTimeline"
    ]
]

gap_df

,Project,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,antsx_antsr,2022-07,2022-10,4,581,3
1,biomedia_mirtk,2022-09,2025-01,29,4,4
2,conda-forge_pyinterp-feedstock,2023-06,2023-09,4,133,3
3,kick-h_gpumd,2018-04,2018-04,1,5837,0
4,mastoffel_rptr,2021-09,2024-08,36,9,8
5,raymanng_3d-wind-field,2021-08,2024-12,41,3,2
6,shader-slang_slang,NaN,NaN,0,0,0
7,theislab_scib,2025-05,2025-07,3,7,1
8,ucl_mimix,2021-09,2021-09,1,21,0
9,worldwidetelescope_wwt-web-client,2023-12,2025-06,19,9,7


In [19]:
part1_stats = summary.groupby("project_wocid").agg(
    ncommits=("commit_sha1", "count"),
    nauthors=("author", "nunique"),
    from_time=("time", "min"),
    to_time=("time", "max")
).reset_index()

part1_stats = part1_stats.rename(
    columns={"project_wocid": "Project"}
)

part1_stats

,Project,ncommits,nauthors,from_time,to_time
0,antsx_antsr,5482,45,1328136237,1762619138
1,biomedia_mirtk,2568,22,1450392347,1741992251
2,conda-forge_pyinterp-feedstock,456,16,1568666567,1761982417
3,kick-h_gpumd,5936,81,1500046377,1760973080
4,mastoffel_rptr,431,11,1414329721,1745921255
5,raymanng_3d-wind-field,267,8,1547622404,1750236118
6,shader-slang_slang,29535,317,1497030169,1761660359
7,theislab_scib,2253,60,1563266409,1761287409
8,ucl_mimix,563,6,1564409694,1636040514
9,worldwidetelescope_wwt-web-client,1416,24,1416030738,1753308292


In [20]:
github_info = {
    "antsx_antsr": {
        "nstars": 143,
        "nforks": 32,
        "lastGHCommitDate": "2026-06-12"
    },

    "biomedia_mirtk": {
        "nstars": 200,
        "nforks": 75,
        "lastGHCommitDate": "2025-02-27"
    },

    "conda-forge_pyinterp-feedstock": {
        "nstars": 0,
        "nforks": 4,
        "lastGHCommitDate": "2025-11-01"
    },

    "kick-h_gpumd": {
        "nstars": 2,
        "nforks": 1,
        "lastGHCommitDate": "2025-10-20"
    },

    "mastoffel_rptr": {
        "nstars": 19,
        "nforks": 1,
        "lastGHCommitDate": "2025-04-29"
    },

    "raymanng_3d-wind-field": {
        "nstars": 497,
        "nforks": 159,
        "lastGHCommitDate": "2025-06-18"
    },

    "shader-slang_slang": {
        "nstars": 5700,
        "nforks": 496,
        "lastGHCommitDate": "2026-09-27"
    },

    "theislab_scib": {
        "nstars": 432,
        "nforks": 78,
        "lastGHCommitDate": "2026-04-27"
    },

    "ucl_mimix": {
        "nstars": 5,
        "nforks": 4,
        "lastGHCommitDate": "2024-09-04"
    },

    "worldwidetelescope_wwt-web-client": {
        "nstars": 108,
        "nforks": 35,
        "lastGHCommitDate": "2026-06-20"
    }
}

In [21]:
github_df = pd.DataFrame.from_dict(
    github_info,
    orient="index"
).reset_index()

github_df = github_df.rename(
    columns={"index": "Project"}
)

In [22]:
stats = part1_stats.merge(
    github_df,
    on="Project",
    how="left"
)

stats = stats.merge(
    gap_df,
    on="Project",
    how="left"
)

stats

,Project,ncommits,nauthors,from_time,to_time,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,antsx_antsr,5482,45,1328136237,1762619138,143,32,2026-06-12,2022-07,2022-10,4,581,3
1,biomedia_mirtk,2568,22,1450392347,1741992251,200,75,2025-02-27,2022-09,2025-01,29,4,4
2,conda-forge_pyinterp-feedstock,456,16,1568666567,1761982417,0,4,2025-11-01,2023-06,2023-09,4,133,3
3,kick-h_gpumd,5936,81,1500046377,1760973080,2,1,2025-10-20,2018-04,2018-04,1,5837,0
4,mastoffel_rptr,431,11,1414329721,1745921255,19,1,2025-04-29,2021-09,2024-08,36,9,8
5,raymanng_3d-wind-field,267,8,1547622404,1750236118,497,159,2025-06-18,2021-08,2024-12,41,3,2
6,shader-slang_slang,29535,317,1497030169,1761660359,5700,496,2026-09-27,NaN,NaN,0,0,0
7,theislab_scib,2253,60,1563266409,1761287409,432,78,2026-04-27,2025-05,2025-07,3,7,1
8,ucl_mimix,563,6,1564409694,1636040514,5,4,2024-09-04,2021-09,2021-09,1,21,0
9,worldwidetelescope_wwt-web-client,1416,24,1416030738,1753308292,108,35,2026-06-20,2023-12,2025-06,19,9,7


In [23]:
activity_patterns = {
    "antsx_antsr": "TODO",
    "biomedia_mirtk": "TODO",
    "conda-forge_pyinterp-feedstock": "TODO",
    "kick-h_gpumd": "TODO",
    "mastoffel_rptr": "TODO",
    "raymanng_3d-wind-field": "TODO",
    "shader-slang_slang": "TODO",
    "theislab_scib": "TODO",
    "ucl_mimix": "TODO",
    "worldwidetelescope_wwt-web-client": "TODO"
}

stats["ActivityPattern"] = stats["Project"].map(activity_patterns)

In [24]:
gap_commit_rows = []

for _, row in stats.iterrows():

    project = row["Project"]

    if pd.isna(row["LongestGapStart"]):
        continue

    gap_start = pd.Period(
        row["LongestGapStart"],
        freq="M"
    )

    gap_end = pd.Period(
        row["LongestGapEnd"],
        freq="M"
    )


    project_commits = summary[
        summary["project_wocid"] == project
    ].copy()

    project_commits = project_commits.sort_values(
        "datetime"
    )


    # Last 10 commits BEFORE gap
    pre = project_commits[
        project_commits["Month"] < gap_start
    ].tail(10).copy()

    pre["pre/post"] = "pre"


    # First 10 commits AFTER gap
    post = project_commits[
        project_commits["Month"] > gap_end
    ].head(10).copy()

    post["pre/post"] = "post"


    selected = pd.concat([pre, post])

    gap_commit_rows.append(selected)
    gap_commits = pd.concat(
    gap_commit_rows,
    ignore_index=True
)

gap_commits = gap_commits[
    [
        "pre/post",
        "project_wocid",
        "commit_sha1",
        "author",
        "time",
        "commit message"
    ]
]

gap_commits.columns = [
    "pre/post",
    "prj",
    "commit",
    "author",
    "time",
    "message"
]

gap_commits.to_csv(
    "kyandall_project_gap_commits.csv",
    sep=";",
    index=False
)

gap_commits

,pre/post,prj,commit,author,time,message
0,pre,antsx_antsr,da9f15aac8a853565c68c00ad613df22ef78adf8,Nick Tustison <ntustison@gmail.com>,1652370714,ENH: Add diffeo.\n
1,pre,antsx_antsr,62d8014e679febdef8b02c3eed68a5d7f0f70053,Nick Tustison <ntustison@gmail.com>,1652373487,ENH: Minor changes.\n
2,pre,antsx_antsr,6d6254f0e7b7fe63159f6637f9481517e1c76452,Nick Tustison <ntustison@gmail.com>,1652451180,DOC/BUG: Compositions not iterations.\n
3,pre,antsx_antsr,5e96f5b12331f2bdb50d4b18998df05c2a20a118,Nick Tustison <ntustison@gmail.com>,1652451561,BUG: One more.\n
4,pre,antsx_antsr,56f08a6df121ef7992d0a5879b3a0e914111aace,Nick Tustison <ntustison@gmail.com>,1652467755,BUG: wrong variable name.\n
...,...,...,...,...,...,...
157,post,worldwidetelescope_wwt-web-client,3c19b689781d647bab3474aefd99b1c3670e3dfc,cranko <cranko@devnull>,1753298551,Release request commit created with Cranko.\n\...
158,post,worldwidetelescope_wwt-web-client,c036f329b017ae1f8928834c7843a0dff9e5dc89,Carifio24 <carifio24@gmail.com>,1753301128,Update deployment workflows to use Ubuntu 22 V...
159,post,worldwidetelescope_wwt-web-client,3ef8e3c6c83c5a36a8b17094bcdf409310143ce9,Jon Carifio <carifio24@gmail.com>,1753301308,Merge pull request #370 from Carifio24/publish...
160,post,worldwidetelescope_wwt-web-client,1f41bca51d0ee112049877f5ddd2d4d03aa94f04,cranko <cranko@devnull>,1753308216,Release request commit created with Cranko.\n\...


In [25]:
def classify_commit(message, author=""):
    text = str(message).lower().strip()
    author = str(author).lower()

    if (
        "dependabot" in text
        or "renovate" in text
        or "github-actions" in author
        or "dependabot" in author
        or "cranko" in author
    ):
        return "Automated bot contributions"

    if (
        "security" in text
        or "cve" in text
        or "vulnerability" in text
    ):
        return "Security patches"

    if (
        "dependency" in text
        or "dependencies" in text
        or "deps" in text
        or "requirements" in text
        or "bump " in text
    ):
        return "Dependency updates"

    if (
        text.startswith("doc")
        or "readme" in text
        or "documentation" in text
    ):
        return "Documentation updates"

    if (
        "release" in text
        or "prepare release" in text
    ):
        return "Release"

    if (
        text.startswith("bug")
        or text.startswith("fix")
        or " bug" in text
        or "fix " in text
        or "fixed" in text
        or "error" in text
        or "crash" in text
    ):
        return "Bug fixes"

    if (
        text.startswith("enh")
        or text.startswith("feat")
        or "feature" in text
        or "implement" in text
        or "add " in text
        or "support " in text
    ):
        return "Feature development"

    return "Other"

In [26]:
gap_commits["Theme"] = gap_commits.apply(
    lambda row: classify_commit(
        row["message"],
        row["author"]
    ),
    axis=1
)

In [27]:
def top_two_themes(df):

    counts = Counter(df["Theme"])

    return ":".join(
        theme
        for theme, count
        in counts.most_common(2)
    )

In [28]:
recent_theme_rows = []

for project in stats["Project"]:

    recent = (
        summary[
            (summary["project_wocid"] == project)
            & (summary["datetime"] <= snapshot_date)
        ]
        .sort_values("datetime")
        .tail(10)
        .copy()
    )

    recent["Theme"] = recent.apply(
        lambda row: classify_commit(
            row["commit message"],
            row["author"]
        ),
        axis=1
    )

    recent_theme_rows.append({
        "Project": project,
        "RecentThemes": top_two_themes(recent)
    })

recent_themes_df = pd.DataFrame(
    recent_theme_rows
)

stats = stats.drop(
    columns=["RecentThemes"],
    errors="ignore"
)

stats = stats.merge(
    recent_themes_df,
    on="Project",
    how="left"
)

NameError: name 'snapshot_date' is not defined

In [ ]:
# Remove any old theme columns from previous runs
cols_to_drop = [
    "BeforeThemes",
    "AfterThemes",
    "BeforeThemes_x",
    "BeforeThemes_y",
    "AfterThemes_x",
    "AfterThemes_y"
]

stats = stats.drop(
    columns=cols_to_drop,
    errors="ignore"
)

# Merge the freshly calculated themes
stats = stats.merge(
    themes_df,
    on="Project",
    how="left"
)

stats[[
    "Project",
    "BeforeThemes",
    "AfterThemes"
]]

In [ ]:
snapshot_date = pd.Timestamp("2025-09-30")
inactive_cutoff = pd.Timestamp("2025-04-01")

snapshot_commits = summary[
    summary["datetime"] <= snapshot_date
]

latest_at_snapshot = (
    snapshot_commits
    .groupby("project_wocid")["datetime"]
    .max()
)

def get_status(project):
    latest = latest_at_snapshot.get(project)

    if pd.isna(latest):
        return "Inactive"

    if latest < inactive_cutoff:
        return "Inactive"

    return "Active"

stats["Currentstatus"] = stats[
    "Project"
].apply(get_status)

In [ ]:
stats.loc[
    stats["Project"] == "shader-slang_slang",
    ["BeforeThemes", "AfterThemes"]
] = "N/A"

In [ ]:
for column in [
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Notes"
]:
    stats[column] = ""

In [ ]:
stats.to_csv(
    "kyandall_project_stats.csv",
    sep=";",
    index=False
)

stats

In [ ]:
recent_theme_rows = []

snapshot_date = pd.Timestamp("2025-09-30")

for project in stats["Project"]:

    recent = (
        summary[
            (summary["project_wocid"] == project)
            & (summary["datetime"] <= snapshot_date)
        ]
        .sort_values("datetime")
        .tail(10)
        .copy()
    )

    recent["Theme"] = recent.apply(
        lambda row: classify_commit(
            row["commit message"],
            row["author"]
        ),
        axis=1
    )

    recent_theme_rows.append({
        "Project": project,
        "RecentThemes": top_two_themes(recent)
    })

recent_themes_df = pd.DataFrame(recent_theme_rows)

stats = stats.drop(
    columns=["RecentThemes"],
    errors="ignore"
)

stats = stats.merge(
    recent_themes_df,
    on="Project",
    how="left"
)

stats[["Project", "RecentThemes"]]

In [ ]:
for project in gap_commits["prj"].unique():

    print("\n")
    print("=" * 80)
    print(project)
    print("=" * 80)

    project_data = gap_commits[
        gap_commits["prj"] == project
    ]

    print("\nBEFORE GAP:")

    for _, row in project_data[
        project_data["pre/post"] == "pre"
    ].iterrows():

        date = pd.to_datetime(
            row["time"],
            unit="s"
        )

        print(
            date.date(),
            "|",
            row["author"],
            "|",
            str(row["message"]).strip()
        )

    print("\nAFTER GAP:")

    for _, row in project_data[
        project_data["pre/post"] == "post"
    ].iterrows():

        date = pd.to_datetime(
            row["time"],
            unit="s"
        )

        print(
            date.date(),
            "|",
            row["author"],
            "|",
            str(row["message"]).strip()
        )

In [29]:
activity_patterns = {
    "antsx_antsr": "irregular",
    "biomedia_mirtk": "declining",
    "conda-forge_pyinterp-feedstock": "irregular",
    "kick-h_gpumd": "rising",
    "mastoffel_rptr": "declining",
    "raymanng_3d-wind-field": "declining",
    "shader-slang_slang": "steady",
    "theislab_scib": "irregular",
    "ucl_mimix": "irregular",
    "worldwidetelescope_wwt-web-client": "irregular"
}

stats["ActivityPattern"] = stats["Project"].map(
    activity_patterns
)

In [30]:
def author_name(author):
    return str(author).split("<")[0].strip()

recovery_people = {}

for project in stats["Project"]:

    project_gap = gap_commits[
        gap_commits["prj"] == project
    ]

    if len(project_gap) == 0:
        recovery_people[project] = "N/A"
        continue

    pre_authors = set(
        project_gap[
            project_gap["pre/post"] == "pre"
        ]["author"].map(author_name)
    )

    post_authors = set(
        project_gap[
            project_gap["pre/post"] == "post"
        ]["author"].map(author_name)
    )

    shared = pre_authors & post_authors
    new = post_authors - pre_authors

    if shared and new:
        recovery_people[project] = (
            "Mixed recovery; returning contributors: "
            + ", ".join(sorted(shared))
            + "; new contributors: "
            + ", ".join(sorted(new))
        )

    elif shared:
        recovery_people[project] = (
            "Primarily returning contributors: "
            + ", ".join(sorted(shared))
        )

    elif post_authors:
        recovery_people[project] = (
            "Primarily new contributors: "
            + ", ".join(sorted(post_authors))
        )

    else:
        recovery_people[project] = "No post-gap commits"

stats["WhoRecovered"] = stats[
    "Project"
].map(recovery_people)

In [31]:
gap_reasons = {

    "antsx_antsr":
        "No explicit cause is stated in the surrounding commits. "
        "Activity stopped after enhancement, bug-fix, documentation, "
        "and compilation work, so the gap appears to be a temporary "
        "maintenance pause rather than abandonment.",

    "biomedia_mirtk":
        "No explicit reason is stated in the surrounding commits. "
        "The 29-month gap followed normal development and maintenance "
        "activity, and only four commits occurred afterward, suggesting "
        "the repository entered a period of very low maintenance.",

    "conda-forge_pyinterp-feedstock":
        "No explicit abandonment signal appears in the commit messages. "
        "The four-month gap appears to be a temporary pause in package "
        "maintenance between groups of update activity.",

    "kick-h_gpumd":
        "The longest gap was only one month and there were no gaps of "
        "three months or longer. This appears to be a normal short pause "
        "rather than meaningful project inactivity.",

    "mastoffel_rptr":
        "No explicit reason for the 36-month gap is stated in the commit "
        "messages. The long period without commits suggests the package "
        "was largely dormant or in low-maintenance mode before development "
        "resumed.",

    "raymanng_3d-wind-field":
        "No explicit cause is given for the 41-month gap. The extremely "
        "long pause followed by only a small number of commits suggests "
        "the project became largely dormant and later received limited "
        "maintenance.",

    "shader-slang_slang":
        "N/A active Project",

    "theislab_scib":
        "No explicit cause is stated in the surrounding commits. The "
        "three-month gap appears to be a temporary development pause "
        "rather than abandonment because activity resumed afterward.",

    "ucl_mimix":
        "The longest gap was only one month, so there is no evidence of "
        "a major interruption at that point. The original project later "
        "became inactive, while its functionality was carried forward "
        "into the RefBasedMI project.",

    "worldwidetelescope_wwt-web-client":
        "The repository experienced a long pause in commits, but there is "
        "no explicit abandonment message. Development later resumed around "
        "release and deployment work."
}

stats["HypothesizedGapReason"] = stats[
    "Project"
].map(gap_reasons)

In [32]:
has_recovered = {
    "antsx_antsr": "Yes",
    "biomedia_mirtk": "Yes, but limited",
    "conda-forge_pyinterp-feedstock": "Yes",
    "kick-h_gpumd": "Yes",
    "mastoffel_rptr": "Yes, but limited",
    "raymanng_3d-wind-field": "Yes, but limited",
    "shader-slang_slang": "N/A",
    "theislab_scib": "Yes",
    "ucl_mimix": "Yes",
    "worldwidetelescope_wwt-web-client": "Yes"
}

why_recovered = {

    "antsx_antsr":
        "Normal development and maintenance resumed after the gap. "
        "There were 581 commits after the longest gap.",

    "biomedia_mirtk":
        "A small amount of maintenance and documentation work resumed, "
        "but only four commits occurred after the longest gap.",

    "conda-forge_pyinterp-feedstock":
        "Package maintenance resumed, including automated update activity. "
        "There were 133 commits after the gap.",

    "kick-h_gpumd":
        "Development resumed immediately after the short pause and continued "
        "at a high level, with 5,837 commits after the gap.",

    "mastoffel_rptr":
        "Maintenance resumed with bug fixes and feature work, although only "
        "nine commits occurred after the long gap.",

    "raymanng_3d-wind-field":
        "The project briefly resumed with release and maintenance-related "
        "work, but only three commits followed the gap.",

    "shader-slang_slang":
        "N/A",

    "theislab_scib":
        "Development resumed after the three-month pause, with seven commits "
        "following the gap in the collected data.",

    "ucl_mimix":
        "Normal development resumed after the one-month pause, with 21 "
        "commits following the gap.",

    "worldwidetelescope_wwt-web-client":
        "Development restarted around deployment and release work. Post-gap "
        "commits include Cranko-generated release commits and deployment "
        "workflow updates."
}

stats["HasRecovered"] = stats[
    "Project"
].map(has_recovered)

stats["WhyRecovered"] = stats[
    "Project"
].map(why_recovered)

In [33]:
reflections = {

    "antsx_antsr":
        """ANTsR shows an irregular activity pattern with three inactivity gaps "
        "of at least three months. Its longest gap lasted four months from "
        "July through October 2022, and the commits immediately before the gap "
        "were normal enhancement, bug-fix, documentation, and compilation work. "
        "There is no obvious explanation for the pause in the commit messages, "
        "but the project clearly recovered because 581 commits occurred afterward.""",

    "biomedia_mirtk":
        """MIRTK shows a declining activity pattern with four gaps of at least "
        "three months. The longest gap lasted 29 months from September 2022 "
        "through January 2025, making it one of the clearest periods of "
        "inactivity in the dataset. The reason is difficult to determine from "
        "the commit messages alone, and the recovery appears weak because only "
        "four commits occurred after the gap.""",

    "conda-forge_pyinterp-feedstock":
        """The pyinterp feedstock has an irregular activity pattern with three "
        "gaps of at least three months. Its longest gap lasted four months from "
        "June through September 2023. The project recovered strongly with 133 "
        "later commits, including automated maintenance activity, so the gap "
        "appears to have been a temporary pause rather than abandonment.""",

    "kick-h_gpumd":
        """gpumd shows a generally rising activity pattern and no inactivity "
        "gaps of at least three months. Its longest period with zero commits was "
        "only one month in April 2018. Because 5,837 commits occurred after that "
        "month, it appears to be a normal short break in an otherwise highly "
        "active project rather than a meaningful interruption.""",

    "mastoffel_rptr":
        """rptR shows a declining and highly intermittent activity pattern, "
        "including eight gaps of at least three months. Its longest gap lasted "
        "36 months from September 2021 through August 2024. The cause is not "
        "explicit in the commit history, but bug fixes and feature work eventually "
        "resumed; with only nine commits afterward, the recovery was limited.""",

    "raymanng_3d-wind-field":
        """3D-Wind-Field shows a declining activity pattern with two substantial "
        "inactivity gaps. Its longest gap lasted 41 months from August 2021 "
        "through December 2024, the longest gap among my assigned projects. "
        "Only three commits followed the gap, so the later activity looks more "
        "like limited maintenance or release work than a full return to active "
        "development.""",

    "shader-slang_slang":
        """Slang shows the most continuous activity of the projects in this "
        "dataset. It had no month with zero commits between its first and last "
        "observed commits, so there is no inactivity gap to interpret. Its "
        "continuous history and large number of contributors support classifying "
        "the project as steady and actively maintained.""",

    "theislab_scib":
        """scib has an irregular activity pattern but only one gap of at least "
        "three months. The gap ran from May through July 2025 and development "
        "resumed afterward. There is no clear abandonment indicator in the "
        "surrounding commits, so this appears to be a temporary development "
        "pause rather than a major project disruption.""",

    "ucl_mimix":
        """mimix had no gap of at least three months in the collected WoC "
        "timeline; its longest zero-commit period was only September 2021. "
        "Twenty-one commits followed that pause, showing that the project "
        "immediately recovered. The original project later became inactive, "
        "and the RefBasedMI repository states that it ports the functionality "
        "of mimix, suggesting that later development moved to a successor project.""",

    "worldwidetelescope_wwt-web-client":
        """The WorldWide Telescope web client shows an irregular pattern with "
        "seven gaps of at least three months. Its longest gap lasted 19 months "
        "from December 2023 through June 2025. Activity resumed with deployment "
        "and release-related work, including Cranko release commits, but only "
        "nine commits followed the gap in the collected data, making the "
        "recovery relatively limited."""
}

In [34]:
for project, reflection in reflections.items():

    filename = f"kyandall_reflection_{project}.md"

    with open(filename, "w") as f:
        f.write(f"# Reflection: {project}\n\n")
        f.write(reflection)

print("Created 10 reflection files.")

Created 10 reflection files.


In [35]:
stats["Notes"] = stats[
    "Project"
].map(reflections)

# Interpretation

- **antsx_antsr — irregular:** Three inactivity gaps of at least three months occurred. The longest was four months, but the project strongly recovered afterward.
- **biomedia_mirtk — declining:** Four substantial gaps occurred, including a 29-month gap, followed by very little activity.
- **conda-forge_pyinterp-feedstock — irregular:** Activity occurs in bursts with three gaps of at least three months, but substantial maintenance activity resumed after the longest gap.
- **kick-h_gpumd — rising:** There were no gaps of at least three months, and most of the project's commits occurred after its only one-month pause.
- **mastoffel_rptr — declining:** Eight substantial inactivity gaps occurred, including a 36-month gap, followed by limited activity.
- **raymanng_3d-wind-field — declining:** The project had a 41-month inactivity gap and only a small amount of activity afterward.
- **shader-slang_slang — steady:** No zero-commit months occurred during the observed timeline.
- **theislab_scib — irregular:** One three-month gap occurred, followed by resumed development.
- **ucl_mimix — irregular:** No gap of at least three months occurred in the observed WoC timeline, although the original project later became inactive.
- **worldwidetelescope_wwt-web-client — irregular:** Seven substantial gaps occurred, including a 19-month gap, followed by limited release and deployment activity.

In [36]:
stats = stats.rename(columns={
    "from_time": "from",
    "to_time": "to"
})

In [37]:
theme_rows = []

for project in gap_commits["prj"].unique():

    project_data = gap_commits[
        gap_commits["prj"] == project
    ]

    pre = project_data[
        project_data["pre/post"] == "pre"
    ].copy()

    post = project_data[
        project_data["pre/post"] == "post"
    ].copy()

    pre["Theme"] = pre.apply(
        lambda row: classify_commit(
            row["message"],
            row["author"]
        ),
        axis=1
    )

    post["Theme"] = post.apply(
        lambda row: classify_commit(
            row["message"],
            row["author"]
        ),
        axis=1
    )

    theme_rows.append({
        "Project": project,
        "BeforeThemes": top_two_themes(pre),
        "AfterThemes": top_two_themes(post)
    })

themes_df = pd.DataFrame(theme_rows)

themes_df = pd.concat([
    themes_df,
    pd.DataFrame([{
        "Project": "shader-slang_slang",
        "BeforeThemes": "N/A",
        "AfterThemes": "N/A"
    }])
], ignore_index=True)

stats = stats.drop(
    columns=[
        "BeforeThemes",
        "AfterThemes",
        "BeforeThemes_x",
        "BeforeThemes_y",
        "AfterThemes_x",
        "AfterThemes_y"
    ],
    errors="ignore"
)

stats = stats.merge(
    themes_df,
    on="Project",
    how="left"
)


snapshot_date = pd.Timestamp("2025-09-30")
inactive_cutoff = pd.Timestamp("2025-04-01")

snapshot_commits = summary[
    summary["datetime"] <= snapshot_date
]

latest_at_snapshot = (
    snapshot_commits
    .groupby("project_wocid")["datetime"]
    .max()
)

def get_status(project):
    latest = latest_at_snapshot.get(project)

    if pd.isna(latest):
        return "Inactive"

    return "Inactive" if latest < inactive_cutoff else "Active"

stats["Currentstatus"] = stats[
    "Project"
].apply(get_status)


recent_theme_rows = []

for project in stats["Project"]:

    recent = (
        summary[
            (summary["project_wocid"] == project)
            & (summary["datetime"] <= snapshot_date)
        ]
        .sort_values("datetime")
        .tail(10)
        .copy()
    )

    recent["Theme"] = recent.apply(
        lambda row: classify_commit(
            row["commit message"],
            row["author"]
        ),
        axis=1
    )

    recent_theme_rows.append({
        "Project": project,
        "RecentThemes": top_two_themes(recent)
    })

recent_themes_df = pd.DataFrame(recent_theme_rows)

stats = stats.drop(
    columns=[
        "RecentThemes",
        "RecentThemes_x",
        "RecentThemes_y"
    ],
    errors="ignore"
)

stats = stats.merge(
    recent_themes_df,
    on="Project",
    how="left"
)


stats["LongestGapStart"] = stats[
    "LongestGapStart"
].fillna("N/A")

stats["LongestGapEnd"] = stats[
    "LongestGapEnd"
].fillna("N/A")


stats = stats.rename(columns={
    "from_time": "from",
    "to_time": "to"
})

print(stats.columns.tolist())

['Project', 'ncommits', 'nauthors', 'from', 'to', 'nstars', 'nforks', 'lastGHCommitDate', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern', 'WhoRecovered', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'Notes', 'BeforeThemes', 'AfterThemes', 'Currentstatus', 'RecentThemes']


In [38]:
final_columns = [
    "Project",
    "ncommits",
    "nauthors",
    "from",
    "to",
    "nstars",
    "nforks",
    "lastGHCommitDate",

    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "ActivityPattern",
    "NumberOfGapsInTimeline",

    "BeforeThemes",
    "AfterThemes",

    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Currentstatus",
    "RecentThemes",
    "Notes"
]

stats = stats[final_columns]

In [39]:
stats = stats[final_columns]

stats.to_csv(
    "kyandall_project_stats.csv",
    sep=";",
    index=False
)

In [40]:
stats.to_csv(
    "kyandall_project_stats.csv",
    sep=";",
    index=False
)

stats

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,...,NumberOfGapsInTimeline,BeforeThemes,AfterThemes,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,antsx_antsr,5482,45,1328136237,1762619138,143,32,2026-06-12,2022-07,2022-10,...,3,Other:Feature development,Feature development:Bug fixes,No explicit cause is stated in the surrounding...,Yes,Normal development and maintenance resumed aft...,Primarily returning contributors: Nick Tustison,Active,Bug fixes:Other,ANTsR shows an irregular activity pattern with...
1,biomedia_mirtk,2568,22,1450392347,1741992251,200,75,2025-02-27,2022-09,2025-01,...,4,Other:Bug fixes,Documentation updates:Other,No explicit reason is stated in the surroundin...,"Yes, but limited",A small amount of maintenance and documentatio...,Primarily returning contributors: Andreas Schuh,Inactive,Other:Bug fixes,MIRTK shows a declining activity pattern with ...
2,conda-forge_pyinterp-feedstock,456,16,1568666567,1761982417,0,4,2025-11-01,2023-06,2023-09,...,3,Other:Bug fixes,Other,No explicit abandonment signal appears in the ...,Yes,"Package maintenance resumed, including automat...",Mixed recovery; returning contributors: Frédér...,Active,Other:Dependency updates,The pyinterp feedstock has an irregular activi...
3,kick-h_gpumd,5936,81,1500046377,1760973080,2,1,2025-10-20,2018-04,2018-04,...,0,Bug fixes:Other,Feature development:Other,The longest gap was only one month and there w...,Yes,Development resumed immediately after the shor...,Primarily returning contributors: Zheyong Fan,Active,Other:Bug fixes,gpumd shows a generally rising activity patter...
4,mastoffel_rptr,431,11,1414329721,1745921255,19,1,2025-04-29,2021-09,2024-08,...,8,Documentation updates:Other,Bug fixes:Other,No explicit reason for the 36-month gap is sta...,"Yes, but limited",Maintenance resumed with bug fixes and feature...,Mixed recovery; returning contributors: Martin...,Active,Bug fixes:Documentation updates,rptR shows a declining and highly intermittent...
5,raymanng_3d-wind-field,267,8,1547622404,1750236118,497,159,2025-06-18,2021-08,2024-12,...,2,Other:Bug fixes,Bug fixes:Other,No explicit cause is given for the 41-month ga...,"Yes, but limited",The project briefly resumed with release and m...,Primarily returning contributors: RaymanNg,Active,Other:Bug fixes,3D-Wind-Field shows a declining activity patte...
6,shader-slang_slang,29535,317,1497030169,1761660359,5700,496,2026-09-27,N/A,N/A,...,0,N/A,N/A,N/A active Project,N/A,N/A,N/A,Active,Other:Feature development,Slang shows the most continuous activity of th...
7,theislab_scib,2253,60,1563266409,1761287409,432,78,2026-04-27,2025-05,2025-07,...,1,Other:Bug fixes,Other,No explicit cause is stated in the surrounding...,Yes,Development resumed after the three-month paus...,Mixed recovery; returning contributors: Leon H...,Active,Other:Dependency updates,scib has an irregular activity pattern but onl...
8,ucl_mimix,563,6,1564409694,1636040514,5,4,2024-09-04,2021-09,2021-09,...,0,Other:Bug fixes,Other:Documentation updates,"The longest gap was only one month, so there i...",Yes,Normal development resumed after the one-month...,"Primarily returning contributors: ianrwhite, m...",Inactive,Other:Documentation updates,mimix had no gap of at least three months in t...
9,worldwidetelescope_wwt-web-client,1416,24,1416030738,1753308292,108,35,2026-06-20,2023-12,2025-06,...,7,Other:Automated bot contributions,Other:Automated bot contributions,The repository experienced a long pause in com...,Yes,Development restarted around deployment and re...,Mixed recovery; returning contributors: Jon Ca...,Active,Automated bot contributions:Other,The WorldWide Telescope web client shows an ir...
